In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, models, Sequential
import matplotlib.pyplot as plt

# 1. Load and Preprocess CIFAR-10
def load_cifar10():
    (x_train, y_train), (x_test, y_test) = tf.keras.datasets.cifar10.load_data()
    # Normalize pixel values to [0, 1]
    x_train, x_test = x_train / 255.0, x_test / 255.0
    return x_train, y_train, x_test, y_test

# 2. Architecture Definitions
def build_lenet(input_shape):
    return Sequential([
        layers.Conv2D(6, (5, 5), activation='tanh', input_shape=input_shape, padding='same'),
        layers.AveragePooling2D(),
        layers.Conv2D(16, (5, 5), activation='tanh'),
        layers.AveragePooling2D(),
        layers.Flatten(),
        layers.Dense(120, activation='tanh'),
        layers.Dense(84, activation='tanh'),
        layers.Dense(10, activation='softmax')
    ], name="LeNet-5")

def build_alexnet(input_shape):
    # Adapted AlexNet for 32x32 input
    return Sequential([
        layers.Conv2D(96, (3, 3), strides=1, activation='relu', input_shape=input_shape),
        layers.BatchNormalization(),
        layers.MaxPooling2D(pool_size=(2, 2)),
        layers.Conv2D(256, (5, 5), padding='same', activation='relu'),
        layers.BatchNormalization(),
        layers.MaxPooling2D(pool_size=(2, 2)),
        layers.Conv2D(384, (3, 3), padding='same', activation='relu'),
        layers.Conv2D(384, (3, 3), padding='same', activation='relu'),
        layers.Conv2D(256, (3, 3), padding='same', activation='relu'),
        layers.MaxPooling2D(pool_size=(2, 2)),
        layers.Flatten(),
        layers.Dense(4096, activation='relu'),
        layers.Dropout(0.5),
        layers.Dense(4096, activation='relu'),
        layers.Dropout(0.5),
        layers.Dense(10, activation='softmax')
    ], name="AlexNet")

def build_vgg16(input_shape):
    # Standard VGG16 via Keras Applications
    return tf.keras.applications.VGG16(weights=None, input_shape=input_shape, classes=10)

def build_resnet50(input_shape):
    # Standard ResNet50 via Keras Applications
    return tf.keras.applications.ResNet50(weights=None, input_shape=input_shape, classes=10)

def build_efficient_separable(input_shape):
    # Efficient architecture using Depthwise Separable Convolutions
    return Sequential([
        layers.SeparableConv2D(32, (3, 3), activation='relu', input_shape=input_shape),
        layers.BatchNormalization(),
        layers.SeparableConv2D(64, (3, 3), activation='relu'),
        layers.MaxPooling2D((2, 2)),
        layers.SeparableConv2D(128, (3, 3), activation='relu'),
        layers.GlobalAveragePooling2D(),
        layers.Dense(64, activation='relu'),
        layers.Dense(10, activation='softmax')
    ], name="Efficient_Separable")

def build_random_features(input_shape):
    # Random Feature Model: Frozen wide hidden layer
    inputs = layers.Input(shape=input_shape)
    x = layers.Flatten()(inputs)
    random_layer = layers.Dense(4096, activation='relu', trainable=False) # Not trained
    x = random_layer(x)
    outputs = layers.Dense(10, activation='softmax')(x)
    return models.Model(inputs, outputs, name="Random_Features")

# 3. Training and Evaluation Routine
def run_experiment():
    x_train, y_train, x_test, y_test = load_cifar10()
    input_shape = (32, 32, 3)
    
    models_list = [
        build_lenet(input_shape),
        build_alexnet(input_shape),
        build_vgg16(input_shape),
        build_resnet50(input_shape),
        build_efficient_separable(input_shape),
        build_random_features(input_shape)
    ]
    
    results = {}

    for model in models_list:
        print(f"\n--- Training {model.name} ---")
        model.compile(optimizer='adam', 
                      loss='sparse_categorical_crossentropy', 
                      metrics=['accuracy'])
        
        # Reduced epochs for demonstration; use 50+ for full convergence
        history = model.fit(x_train, y_train, epochs=5, batch_size=128, 
                            validation_data=(x_test, y_test), verbose=1)
        
        results[model.name] = {
            'accuracy': history.history['val_accuracy'][-1],
            'params': model.count_params()
        }

    # Final Comparison Summary
    print("\n" + "="*30)
    print(f"{'Architecture':<20} | {'Params':<10} | {'Test Acc':<8}")
    print("-" * 45)
    for name, data in results.items():
        print(f"{name:<20} | {data['params']:<10,} | {data['accuracy']:.4f}")

if __name__ == "__main__":
    run_experiment()


C:\Users\user\anaconda3\Lib\site-packages\pandas\core\arrays\masked.py:61: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.5' currently installed).
  from pandas.core import (





--- Training LeNet-5 ---

Epoch 1/5


391/391 [==============================] - 8s 16ms/step - loss: 1.7929 - accuracy: 0.3689 - val_loss: 1.6783 - val_accuracy: 0.4098
Epoch 2/5
391/391 [==============================] - 6s 15ms/step - loss: 1.6111 - accuracy: 0.4335 - val_loss: 1.5508 - val_accuracy: 0.4475
Epoch 3/5
391/391 [==============================] - 6s 15ms/step - loss: 1.4786 - accuracy: 0.4761 - val_loss: 1.4354 - val_accuracy: 0.4883
Epoch 4/5
391/391 [==============================] - 6s 16ms/step - loss: 1.3870 - accuracy: 0.5088 - val_loss: 1.3722 - val_accuracy: 0.5176
Epoch 5/5
391/391 [==============================] - 6s 15ms/step - loss: 1.3153 - accuracy: 0.5341 - val_loss: 1.3338 - val_accuracy: 0.5307

--- Training AlexNet ---
Epoch 1/5
391/391 [==============================] - 717s 2s/step - loss: 1.7050 - accuracy: 0.3790 - val_loss: 2.6439 - val_accuracy: 0.2221
Epoch 2/5
391/391 [==============================] - 713s 2s/step - loss: 1.1578 - accuracy